# Tamil Nadu FY 2025-26 peak-week optimization at 15-minute resolution

This notebook loads the full-year 15-minute input dataset, selects the Monday-Sunday week containing the annual demand peak, optimizes its 672 quarter-hour snapshots with HiGHS, and exports the solved week to a separate NetCDF file.

> The 15-minute mixed-integer solve is substantially larger than the 168-snapshot hourly version and may take considerably longer.

In [1]:
from pathlib import Path

import pandas as pd
import pypsa

INPUT_DIR = Path("inputs") / "tamil_nadu_ra_2025_26_15min"
OUTPUT_FILE = Path("tamil_nadu_2025_26_15min.nc")

n_year = pypsa.Network(INPUT_DIR)
component_metadata = pd.read_csv(INPUT_DIR / "component_metadata.csv")
plant_generators = component_metadata.loc[
    component_metadata.component_type == "Generator", "component_id"
]
plant_storage = component_metadata.loc[
    component_metadata.component_type == "StorageUnit", "component_id"
]

assert set(plant_generators) <= set(n_year.generators.index)
assert set(plant_storage) == set(n_year.storage_units.index)
assert len(n_year.snapshots) == 35_040
assert (n_year.snapshots.to_series().diff().dropna() == pd.Timedelta(minutes=15)).all()
assert (n_year.snapshot_weightings[["objective", "stores", "generators"]] == 0.25).all().all()

print(
    f"Loaded {len(plant_generators)} workbook-record generators, "
    f"{len(plant_storage)} storage units, and {len(n_year.snapshots):,} snapshots"
)

INFO:pypsa.network.io:Imported network 'Tamil Nadu FY 2025-26 single-node UC inputs (15-minute)' has buses, carriers, generators, loads, storage_units


Loaded 276 workbook-record generators, 4 storage units, and 35,040 snapshots


In [2]:
# Select the Monday-Sunday week containing the annual demand peak.
annual_demand = n_year.loads_t.p_set.sum(axis=1)
peak_timestamp = annual_demand.idxmax()
week_start = peak_timestamp.normalize() - pd.Timedelta(days=peak_timestamp.weekday())
week_end = week_start + pd.Timedelta(days=7)
week = n_year.snapshots[(n_year.snapshots >= week_start) & (n_year.snapshots < week_end)]

assert len(week) == 672, f"Expected 672 quarter-hour snapshots, found {len(week)}"
n = n_year.copy(snapshots=week)

print(f"Annual peak: {annual_demand.loc[peak_timestamp]:,.2f} MW at {peak_timestamp}")
print(
    f"Optimizing peak week: {week_start} to "
    f"{week_end - pd.Timedelta(minutes=15)} ({len(week)} snapshots)"
)

Annual peak: 19,987.33 MW at 2025-07-11 16:00:00
Optimizing peak week: 2025-07-07 00:00:00 to 2025-07-13 23:45:00 (672 snapshots)


In [3]:
n

PyPSA Network 'Tamil Nadu FY 2025-26 single-node UC inputs (15-minute)'
-----------------------------------------------------------------------
Components:
 - Bus: 1
 - Carrier: 12
 - Generator: 278
 - Load: 1
 - StorageUnit: 4
Snapshots: 672

## Optimize the 672-snapshot peak week

In [4]:
status, termination_condition = n.optimize(
    solver_name="highs",
    formulation="kirchhoff",
)

if termination_condition != "optimal":
    raise RuntimeError(f"Optimization failed: {status}, {termination_condition}")

print(f"Optimization: {status} ({termination_condition})")

C:\Users\b076218\AppData\Local\Temp\ipykernel_17892\3191222579.py:1: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  status, termination_condition = n.optimize(
       'mettur_tps__unit_01', 'mettur_tps__unit_02', 'mettur_tps__unit_03',
       'mettur_tps__unit_04', 'mettur_tps__unit_05', 'muthiara_tpp__unit_01',
       'muthiara_tpp__unit_02', 'neyveli_ext_tps_i__unit_01',
       ...
       'shakti_sugars_ltd_tamil_nadu_erode_32__unit_01',
       'shriraam_city_union_finance_ltd_tamil_nadu_thanjavur_7_5__unit_01',
       'shriram_investments_ltd_tamil_nadu_dindigul_7_5__unit_01',
       'sripathi_paper_board_p_ltd_tamil_nadu_virudhunagar_4_95_09_03_20__unit_01',
       'subashri_bio_energy_p_ltd_tamil_nadu_namakkal_2_5__unit_01',
       'subramania_si

Optimization: ok (optimal)


In [5]:
n.export_to_netcdf(OUTPUT_FILE)
print(f"Exported solved peak week to {OUTPUT_FILE}")

INFO:pypsa.network.io:Exported network 'Tamil Nadu FY 2025-26 single-node UC inputs (15-minute)' saved to 'tamil_nadu_2025_26_15min.nc contains: sub_networks, buses, loads, storage_units, carriers, generators


Exported solved peak week to tamil_nadu_2025_26_15min.nc
